<a href="https://colab.research.google.com/github/riadimrt/ACOAlgorithm/blob/main/Konvergensi%20SAW%20AHP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
BAB 1: Demonstrasi Simple Additive Weighting (SAW)
Studi Kasus Baku Buku: Pemilihan Vendor Katering untuk Acara Kantor
10 Vendor (alternatif) x 7 Kriteria (4 benefit, 3 cost)

Alur: Input -> Normalisasi (benefit & cost) -> Pembobotan -> Skor Akhir -> Peringkat

Dataset ini menjadi acuan TETAP di seluruh buku "SAW dan Algoritma yang
Konvergen Dengannya". Setiap bab berikutnya (MCE-AHP, MAVT, SMART, ARAS,
WASPAS, WPM, TOPSIS) akan memakai angka masukan yang persis sama ini,
sehingga pembaca bisa membandingkan hasil akhirnya secara langsung.
"""

# ============================================================
# LANGKAH 1: INPUT DATA (MATRIKS KEPUTUSAN)
# ============================================================
# Tujuh kriteria penilaian vendor katering:
#   4 kriteria BENEFIT (semakin besar semakin baik)
#     K1 = Kualitas Rasa              (skor uji tester, 1-100)
#     K2 = Ketepatan Waktu            (skor rekam jejak pengiriman, 1-100)
#     K3 = Variasi Menu               (skor keragaman pilihan menu, 1-100)
#     K4 = Rating Kepuasan Pelanggan  (skor ulasan, 1-100)
#   3 kriteria COST (semakin kecil semakin baik)
#     K5 = Harga per Porsi            (Rupiah)
#     K6 = Lead Time Persiapan        (hari)
#     K7 = Jumlah Komplain per Event  (kali)

vendor = [
    "Nirmala Catering", "Cita Rasa Boga", "Sajian Prima", "Meja Utama",
    "Berkah Boga", "Dapur Bunda", "Rasa Nusantara", "Prima Boga",
    "Sinar Catering", "Boga Sejahtera",
]

kriteria = [
    "Kualitas Rasa", "Ketepatan Waktu", "Variasi Menu", "Rating Kepuasan",
    "Harga/Porsi", "Lead Time", "Jml Komplain",
]

tipe = ["benefit", "benefit", "benefit", "benefit", "cost", "cost", "cost"]

# Matriks keputusan: key = nama vendor, value = [K1..K7]
data = {
    "Nirmala Catering": [85, 78, 70, 80, 35000, 3, 2],
    "Cita Rasa Boga":   [90, 70, 65, 75, 42000, 2, 1],
    "Sajian Prima":     [75, 85, 80, 85, 38000, 4, 3],
    "Meja Utama":       [80, 80, 75, 82, 36000, 3, 1],
    "Berkah Boga":      [70, 90, 85, 78, 30000, 5, 4],
    "Dapur Bunda":      [78, 75, 60, 70, 28000, 3, 2],
    "Rasa Nusantara":   [88, 65, 90, 88, 45000, 2, 1],
    "Prima Boga":       [72, 82, 55, 65, 27000, 4, 3],
    "Sinar Catering":   [83, 88, 70, 80, 33000, 3, 1],
    "Boga Sejahtera":   [68, 60, 50, 60, 25000, 5, 5],
}

# Bobot kriteria (hasil AHP, diturunkan langkah demi langkah di Bab 3)
bobot = [0.20, 0.15, 0.10, 0.15, 0.20, 0.10, 0.10]

n = len(kriteria)


def cetak_tabel(judul, sumber, presisi=None):
    print("\n" + "=" * 100)
    print(judul)
    print("=" * 100)
    header = f"{'Vendor':<18}" + "".join(f"{k:>13}" for k in kriteria)
    print(header)
    for v in vendor:
        if presisi is None:
            row = f"{v:<18}" + "".join(f"{sumber[v][j]:>13}" for j in range(n))
        else:
            row = f"{v:<18}" + "".join(f"{round(sumber[v][j], presisi):>13}" for j in range(n))
        print(row)


cetak_tabel("LANGKAH 1: DATA MASUKAN (MATRIKS KEPUTUSAN)", data)
print(f"\nTipe kriteria  : {list(zip(kriteria, tipe))}")
print(f"Bobot kriteria : {bobot}  (total = {round(sum(bobot), 4)})")

# ============================================================
# LANGKAH 2: NORMALISASI
# Benefit : r_ij = x_ij / nilai_maksimum_kolom_j
# Cost    : r_ij = nilai_minimum_kolom_j / x_ij
# ============================================================
maks_kolom = [max(data[v][j] for v in vendor) for j in range(n)]
min_kolom = [min(data[v][j] for v in vendor) for j in range(n)]

normal = {}
for v in vendor:
    baris = []
    for j in range(n):
        if tipe[j] == "benefit":
            baris.append(round(data[v][j] / maks_kolom[j], 4))
        else:
            baris.append(round(min_kolom[j] / data[v][j], 4))
    normal[v] = baris

print("\n" + "=" * 100)
print("LANGKAH 2: NORMALISASI (benefit: x/maks | cost: min/x)")
print("=" * 100)
print(f"Nilai maksimum kolom : {maks_kolom}")
print(f"Nilai minimum kolom  : {min_kolom}")
cetak_tabel("Matriks Ternormalisasi", normal)

# ============================================================
# LANGKAH 3: PEMBOBOTAN DAN PENJUMLAHAN (SKOR SAW)
# Rumus: V_i = sum_j (bobot_j * r_ij)
# ============================================================
skor = {}
for v in vendor:
    skor[v] = sum(normal[v][j] * bobot[j] for j in range(n))

print("\n" + "=" * 100)
print("LANGKAH 3: SKOR AKHIR SAW (nilai ternormalisasi dikali bobot, dijumlahkan)")
print("=" * 100)
for v in vendor:
    detail = " + ".join(f"({normal[v][j]}x{bobot[j]})" for j in range(n))
    print(f"{v}")
    print(f"  {detail}")
    print(f"  = {round(skor[v], 4)}")

# ============================================================
# LANGKAH 4: PERINGKAT AKHIR
# ============================================================
peringkat = sorted(vendor, key=lambda v: skor[v], reverse=True)

print("\n" + "=" * 100)
print("LANGKAH 4: PERINGKAT VENDOR (skor tertinggi = terbaik)")
print("=" * 100)
for i, v in enumerate(peringkat, start=1):
    print(f"{i:>2}. {v:<18} skor = {round(skor[v], 4)}")

print(f"\n>>> Vendor terpilih (metode SAW): {peringkat[0]} <<<")



LANGKAH 1: DATA MASUKAN (MATRIKS KEPUTUSAN)
Vendor            Kualitas RasaKetepatan Waktu Variasi MenuRating Kepuasan  Harga/Porsi    Lead Time Jml Komplain
Nirmala Catering             85           78           70           80        35000            3            2
Cita Rasa Boga               90           70           65           75        42000            2            1
Sajian Prima                 75           85           80           85        38000            4            3
Meja Utama                   80           80           75           82        36000            3            1
Berkah Boga                  70           90           85           78        30000            5            4
Dapur Bunda                  78           75           60           70        28000            3            2
Rasa Nusantara               88           65           90           88        45000            2            1
Prima Boga                   72           82           55           65 

In [ ]:
"""
BAB 3 - WSM dan MCE-AHP: Saudara Kembar SAW
Studi Kasus Baku Buku: Pemilihan Vendor Katering (10 vendor x 7 kriteria)

Alur:
  Bagian A - WSM      : membuktikan WSM identik dengan SAW (data & bobot sama persis)
  Bagian B - AHP      : menyusun matriks perbandingan berpasangan 7 kriteria,
                        menghitung bobot (eigenvector approx) dan CR
  Bagian C - MCE-AHP  : menghitung skor pakai matriks ternormalisasi yang sama,
                        bobot dari hasil AHP
  Bagian D - Bandingkan: papan skor kumulatif SAW vs WSM vs MCE-AHP
"""

# ============================================================
# DATA DASAR (identik dengan Bab 1 dan Bab 2, JANGAN diubah)
# ============================================================
vendor = [
    "Nirmala Catering", "Cita Rasa Boga", "Sajian Prima", "Meja Utama",
    "Berkah Boga", "Dapur Bunda", "Rasa Nusantara", "Prima Boga",
    "Sinar Catering", "Boga Sejahtera",
]

kriteria = [
    "Kualitas Rasa", "Ketepatan Waktu", "Variasi Menu", "Rating Kepuasan",
    "Harga/Porsi", "Lead Time", "Jml Komplain",
]

tipe = ["benefit", "benefit", "benefit", "benefit", "cost", "cost", "cost"]

data = {
    "Nirmala Catering": [85, 78, 70, 80, 35000, 3, 2],
    "Cita Rasa Boga":   [90, 70, 65, 75, 42000, 2, 1],
    "Sajian Prima":     [75, 85, 80, 85, 38000, 4, 3],
    "Meja Utama":       [80, 80, 75, 82, 36000, 3, 1],
    "Berkah Boga":      [70, 90, 85, 78, 30000, 5, 4],
    "Dapur Bunda":      [78, 75, 60, 70, 28000, 3, 2],
    "Rasa Nusantara":   [88, 65, 90, 88, 45000, 2, 1],
    "Prima Boga":       [72, 82, 55, 65, 27000, 4, 3],
    "Sinar Catering":   [83, 88, 70, 80, 33000, 3, 1],
    "Boga Sejahtera":   [68, 60, 50, 60, 25000, 5, 5],
}

bobot_bab2 = [0.20, 0.15, 0.10, 0.15, 0.20, 0.10, 0.10]
n = len(kriteria)


def normalisasi(matriks_data):
    maks = [max(matriks_data[v][j] for v in vendor) for j in range(n)]
    mins = [min(matriks_data[v][j] for v in vendor) for j in range(n)]
    hasil = {}
    for v in vendor:
        baris = []
        for j in range(n):
            if tipe[j] == "benefit":
                baris.append(round(matriks_data[v][j] / maks[j], 4))
            else:
                baris.append(round(mins[j] / matriks_data[v][j], 4))
        hasil[v] = baris
    return hasil


def hitung_skor(matriks_normal, bobot):
    return {v: round(sum(matriks_normal[v][j] * bobot[j] for j in range(n)), 4) for v in vendor}


def cetak_peringkat(judul, skor):
    print("\n" + "-" * 70)
    print(judul)
    print("-" * 70)
    for i, v in enumerate(sorted(vendor, key=lambda x: skor[x], reverse=True), 1):
        print(f"{i:>2}. {v:<18} skor = {skor[v]}")


matriks_normal = normalisasi(data)

# ============================================================
# BAGIAN A: WSM (Weighted Sum Model)
# ============================================================
print("=" * 70)
print("BAGIAN A: WSM - MEMBUKTIKAN IDENTIK DENGAN SAW")
print("=" * 70)
print("WSM memakai matriks ternormalisasi dan bobot yang PERSIS SAMA dengan SAW.")
print("Tidak ada langkah tambahan, tidak ada rumus berbeda.")

skor_saw = hitung_skor(matriks_normal, bobot_bab2)
skor_wsm = hitung_skor(matriks_normal, bobot_bab2)  # rumus identik

print(f"\n{'Vendor':<20}{'Skor SAW':>12}{'Skor WSM':>12}{'Selisih':>12}")
for v in vendor:
    selisih = round(skor_saw[v] - skor_wsm[v], 6)
    print(f"{v:<20}{skor_saw[v]:>12}{skor_wsm[v]:>12}{selisih:>12}")
print("\n>>> Selisih SAW vs WSM = 0 untuk seluruh vendor (K = 1). <<<")

# ============================================================
# BAGIAN B: AHP - MATRIKS PERBANDINGAN BERPASANGAN & BOBOT
# ============================================================
print("\n" + "=" * 70)
print("BAGIAN B: AHP - PENURUNAN BOBOT DARI PERBANDINGAN BERPASANGAN")
print("=" * 70)

# Pengelompokan tingkat kepentingan kriteria:
#   G1 (paling penting) = Kualitas Rasa, Harga/Porsi
#   G2 (menengah)       = Ketepatan Waktu, Rating Kepuasan
#   G3 (pendukung)      = Variasi Menu, Lead Time, Jml Komplain
grup = {0: "G1", 4: "G1", 1: "G2", 3: "G2", 2: "G3", 5: "G3", 6: "G3"}
level = {"G1": 3, "G2": 2, "G3": 1}
v1, v2 = 1.5, 2.0  # nilai Saaty: selisih 1 level = 1.5 | selisih 2 level = 2.0

M = [[1.0] * n for _ in range(n)]
for i in range(n):
    for j in range(n):
        if i == j:
            continue
        li, lj = level[grup[i]], level[grup[j]]
        if li - lj == 1:
            M[i][j] = v1
        elif li - lj == 2:
            M[i][j] = v2
        elif lj - li == 1:
            M[i][j] = 1 / v1
        elif lj - li == 2:
            M[i][j] = 1 / v2

print("Matriks Perbandingan Berpasangan (7x7):")
header = f"{'':<18}" + "".join(f"{k[:8]:>10}" for k in kriteria)
print(header)
for i in range(n):
    row = f"{kriteria[i]:<18}" + "".join(f"{round(M[i][j],3):>10}" for j in range(n))
    print(row)

# Bobot: normalisasi kolom lalu rata-rata baris (pendekatan eigenvector)
jumlah_kolom = [sum(M[i][j] for i in range(n)) for j in range(n)]
M_norm = [[M[i][j] / jumlah_kolom[j] for j in range(n)] for i in range(n)]
bobot_ahp = [round(sum(M_norm[i][j] for j in range(n)) / n, 4) for i in range(n)]

print(f"\nBobot AHP hasil perhitungan:")
for k, w in zip(kriteria, bobot_ahp):
    print(f"  {k:<18}: {w}")
print(f"  Total            : {round(sum(bobot_ahp),4)}")

# Verifikasi konsistensi (CR)
Mw = [sum(M[i][j] * bobot_ahp[j] for j in range(n)) for i in range(n)]
lam_i = [Mw[i] / bobot_ahp[i] for i in range(n)]
lambda_max = sum(lam_i) / n
CI = (lambda_max - n) / (n - 1)
RI = 1.32  # tabel RI Saaty untuk n = 7
CR = CI / RI

print(f"\nlambda_max = {round(lambda_max,4)}")
print(f"CI         = {round(CI,4)}")
print(f"RI (n=7)   = {RI}")
print(f"CR         = {round(CR,4)} ({round(CR*100,2)}%)")
status = "KONSISTEN (CR <= 10%)" if CR <= 0.10 else "TIDAK KONSISTEN (CR > 10%)"
print(f"Status     : {status}")

# ============================================================
# BAGIAN C: MCE-AHP - SKOR DENGAN BOBOT DARI AHP
# ============================================================
print("\n" + "=" * 70)
print("BAGIAN C: MCE-AHP - SKOR MEMAKAI MATRIKS TERNORMALISASI YANG SAMA")
print("=" * 70)
print("(matriks ternormalisasi identik dengan Tabel 2.2 di Bab 2, hanya bobot berganti)")

skor_mce = hitung_skor(matriks_normal, bobot_ahp)
cetak_peringkat("Peringkat MCE-AHP", skor_mce)

# ============================================================
# BAGIAN D: PAPAN SKOR KUMULATIF
# ============================================================
print("\n" + "=" * 70)
print("BAGIAN D: PAPAN SKOR KUMULATIF - SAW vs WSM vs MCE-AHP")
print("=" * 70)
peringkat_saw = sorted(vendor, key=lambda v: skor_saw[v], reverse=True)
peringkat_mce = sorted(vendor, key=lambda v: skor_mce[v], reverse=True)

print(f"{'Peringkat':<10}{'SAW':<20}{'WSM':<20}{'MCE-AHP':<20}{'Urutan Sama?':<12}")
sama_semua = True
for i in range(len(vendor)):
    v_saw, v_mce = peringkat_saw[i], peringkat_mce[i]
    sama = "Ya" if v_saw == v_mce else "TIDAK"
    if v_saw != v_mce:
        sama_semua = False
    print(f"{i+1:<10}{v_saw:<20}{v_saw:<20}{v_mce:<20}{sama:<12}")

print(f"\n>>> Seluruh peringkat identik antara SAW, WSM, dan MCE-AHP: {sama_semua} <<<")

BAGIAN A: WSM - MEMBUKTIKAN IDENTIK DENGAN SAW
WSM memakai matriks ternormalisasi dan bobot yang PERSIS SAMA dengan SAW.
Tidak ada langkah tambahan, tidak ada rumus berbeda.

Vendor                  Skor SAW    Skor WSM     Selisih
Nirmala Catering          0.7926      0.7926         0.0
Cita Rasa Boga            0.8358      0.8358         0.0
Sajian Prima               0.757       0.757         0.0
Meja Utama                0.8398      0.8398         0.0
Berkah Boga               0.7646      0.7646         0.0
Dapur Bunda               0.7796      0.7796         0.0
Rasa Nusantara             0.865       0.865         0.0
Prima Boga                0.7371      0.7371         0.0
Sinar Catering            0.8634      0.8634         0.0
Boga Sejahtera             0.669       0.669         0.0

>>> Selisih SAW vs WSM = 0 untuk seluruh vendor (K = 1). <<<

BAGIAN B: AHP - PENURUNAN BOBOT DARI PERBANDINGAN BERPASANGAN
Matriks Perbandingan Berpasangan (7x7):
                    Kualitas  Ket

In [ ]:
"""
BAB 4 - MAVT dan SMART: Beda Cara Timbang, Sama Hasil
Studi Kasus Baku Buku: Pemilihan Vendor Katering (10 vendor x 7 kriteria)

Alur:
  Bagian A - MAVT     : fungsi nilai proporsional (identik dengan normalisasi SAW),
                        bobot dipakai sama seperti Bab 2 -> skor MAVT
  Bagian B - SMART    : swing weighting untuk memperoleh bobot baru,
                        dihitung dari matriks ternormalisasi yang sama -> skor SMART
  Bagian C - Bandingkan: papan skor kumulatif SAW, MCE-AHP, MAVT, SMART
"""

# ============================================================
# DATA DASAR (identik dengan Bab 1, 2, 3 - JANGAN diubah)
# ============================================================
vendor = [
    "Nirmala Catering", "Cita Rasa Boga", "Sajian Prima", "Meja Utama",
    "Berkah Boga", "Dapur Bunda", "Rasa Nusantara", "Prima Boga",
    "Sinar Catering", "Boga Sejahtera",
]

kriteria = [
    "Kualitas Rasa", "Ketepatan Waktu", "Variasi Menu", "Rating Kepuasan",
    "Harga/Porsi", "Lead Time", "Jml Komplain",
]

tipe = ["benefit", "benefit", "benefit", "benefit", "cost", "cost", "cost"]

data = {
    "Nirmala Catering": [85, 78, 70, 80, 35000, 3, 2],
    "Cita Rasa Boga":   [90, 70, 65, 75, 42000, 2, 1],
    "Sajian Prima":     [75, 85, 80, 85, 38000, 4, 3],
    "Meja Utama":       [80, 80, 75, 82, 36000, 3, 1],
    "Berkah Boga":      [70, 90, 85, 78, 30000, 5, 4],
    "Dapur Bunda":      [78, 75, 60, 70, 28000, 3, 2],
    "Rasa Nusantara":   [88, 65, 90, 88, 45000, 2, 1],
    "Prima Boga":       [72, 82, 55, 65, 27000, 4, 3],
    "Sinar Catering":   [83, 88, 70, 80, 33000, 3, 1],
    "Boga Sejahtera":   [68, 60, 50, 60, 25000, 5, 5],
}

bobot_bab2 = [0.20, 0.15, 0.10, 0.15, 0.20, 0.10, 0.10]
n = len(kriteria)


def normalisasi(matriks_data):
    """Fungsi nilai proporsional: benefit x/maks, cost min/x (x_j- = 0)."""
    maks = [max(matriks_data[v][j] for v in vendor) for j in range(n)]
    mins = [min(matriks_data[v][j] for v in vendor) for j in range(n)]
    hasil = {}
    for v in vendor:
        baris = []
        for j in range(n):
            if tipe[j] == "benefit":
                baris.append(round(matriks_data[v][j] / maks[j], 4))
            else:
                baris.append(round(mins[j] / matriks_data[v][j], 4))
        hasil[v] = baris
    return hasil


def hitung_skor(matriks_normal, bobot):
    return {v: round(sum(matriks_normal[v][j] * bobot[j] for j in range(n)), 4) for v in vendor}


def cetak_peringkat(judul, skor):
    print("\n" + "-" * 70)
    print(judul)
    print("-" * 70)
    for i, v in enumerate(sorted(vendor, key=lambda x: skor[x], reverse=True), 1):
        print(f"{i:>2}. {v:<18} skor = {skor[v]}")


matriks_normal = normalisasi(data)

# ============================================================
# BAGIAN A: MAVT - FUNGSI NILAI PROPORSIONAL
# ============================================================
print("=" * 70)
print("BAGIAN A: MAVT - FUNGSI NILAI PROPORSIONAL (x_j- = 0)")
print("=" * 70)
print("Fungsi nilai: benefit -> x/maks | cost -> min/x")
print("(identik dengan normalisasi SAW pada Bab 2, sehingga hasilnya pun identik)")

skor_saw = hitung_skor(matriks_normal, bobot_bab2)
skor_mavt = hitung_skor(matriks_normal, bobot_bab2)  # fungsi nilai & bobot identik

print(f"\n{'Vendor':<20}{'Skor SAW':>12}{'Skor MAVT':>12}{'Selisih':>12}")
for v in vendor:
    selisih = round(skor_saw[v] - skor_mavt[v], 6)
    print(f"{v:<20}{skor_saw[v]:>12}{skor_mavt[v]:>12}{selisih:>12}")
print("\n>>> Selisih SAW vs MAVT = 0 untuk seluruh vendor (K = 1). <<<")

# ============================================================
# BAGIAN B: SMART - SWING WEIGHTING
# ============================================================
print("\n" + "=" * 70)
print("BAGIAN B: SMART - BOBOT DARI SWING WEIGHTING")
print("=" * 70)

# Poin swing (0-100) dari penilaian tim pengadaan: seberapa besar dampak
# perpindahan dari kondisi terburuk ke terbaik pada tiap kriteria.
swing_poin = {
    "Kualitas Rasa": 100,
    "Harga/Porsi": 100,
    "Ketepatan Waktu": 70,
    "Rating Kepuasan": 70,
    "Variasi Menu": 48,
    "Lead Time": 48,
    "Jml Komplain": 48,
}
poin_urut = [swing_poin[k] for k in kriteria]
total_poin = sum(poin_urut)
bobot_smart = [round(p / total_poin, 4) for p in poin_urut]

print("Poin swing weighting per kriteria:")
for k in kriteria:
    print(f"  {k:<18}: {swing_poin[k]}")
print(f"  Total poin       : {total_poin}")

print(f"\nBobot SMART hasil normalisasi poin:")
for k, w in zip(kriteria, bobot_smart):
    print(f"  {k:<18}: {w}")
print(f"  Total            : {round(sum(bobot_smart),4)}")

print("\nCatatan: SMART tidak memerlukan matriks perbandingan berpasangan,")
print("tidak ada lambda_max, dan tidak ada pemeriksaan Consistency Ratio (CR).")

skor_smart = hitung_skor(matriks_normal, bobot_smart)
cetak_peringkat("Peringkat SMART", skor_smart)

# ============================================================
# BAGIAN C: PAPAN SKOR KUMULATIF (4 METODE)
# ============================================================
print("\n" + "=" * 70)
print("BAGIAN C: PAPAN SKOR KUMULATIF - SAW, MCE-AHP, MAVT, SMART")
print("=" * 70)

# Bobot AHP dari Bab 3 (untuk kolom MCE-AHP)
bobot_ahp = [0.2061, 0.1445, 0.0996, 0.1445, 0.2061, 0.0996, 0.0996]
skor_mce = hitung_skor(matriks_normal, bobot_ahp)

peringkat_saw = sorted(vendor, key=lambda v: skor_saw[v], reverse=True)

print(f"{'Rank':<6}{'Vendor':<20}{'SAW':>10}{'MCE-AHP':>10}{'MAVT':>10}{'SMART':>10}")
for i, v in enumerate(peringkat_saw, 1):
    print(f"{i:<6}{v:<20}{skor_saw[v]:>10}{skor_mce[v]:>10}{skor_mavt[v]:>10}{skor_smart[v]:>10}")

# Verifikasi urutan sama di keempat metode
urutan_sama = True
for metode_skor in [skor_mce, skor_mavt, skor_smart]:
    urutan_metode = sorted(vendor, key=lambda v: metode_skor[v], reverse=True)
    if urutan_metode != peringkat_saw:
        urutan_sama = False

print(f"\n>>> Urutan peringkat identik di keempat metode (SAW/MCE-AHP/MAVT/SMART): {urutan_sama} <<<")

BAGIAN A: MAVT - FUNGSI NILAI PROPORSIONAL (x_j- = 0)
Fungsi nilai: benefit -> x/maks | cost -> min/x
(identik dengan normalisasi SAW pada Bab 2, sehingga hasilnya pun identik)

Vendor                  Skor SAW   Skor MAVT     Selisih
Nirmala Catering          0.7926      0.7926         0.0
Cita Rasa Boga            0.8358      0.8358         0.0
Sajian Prima               0.757       0.757         0.0
Meja Utama                0.8398      0.8398         0.0
Berkah Boga               0.7646      0.7646         0.0
Dapur Bunda               0.7796      0.7796         0.0
Rasa Nusantara             0.865       0.865         0.0
Prima Boga                0.7371      0.7371         0.0
Sinar Catering            0.8634      0.8634         0.0
Boga Sejahtera             0.669       0.669         0.0

>>> Selisih SAW vs MAVT = 0 untuk seluruh vendor (K = 1). <<<

BAGIAN B: SMART - BOBOT DARI SWING WEIGHTING
Poin swing weighting per kriteria:
  Kualitas Rasa     : 100
  Ketepatan Waktu   : 70


In [ ]:
"""
BAB 5 - ARAS dan WASPAS: Variasi Rasio dan Kombinasi
Studi Kasus Baku Buku: Pemilihan Vendor Katering (10 vendor x 7 kriteria)

Alur:
  Bagian A - ARAS (varian praktis)  : S_i dihitung dari matriks ternormalisasi
                                      yang sama seperti SAW, S_0 = 1 (alternatif
                                      optimal), K_i = S_i / S_0
  Bagian A2 - ARAS (varian asli)    : normalisasi jumlah kolom ala Zavadskas,
                                      ditampilkan untuk kejujuran metodologis
  Bagian B - WASPAS (lambda=1)      : identitas aljabar, Q_i = V_i^SAW
  Bagian C - Papan skor kumulatif   : 6 metode (SAW, MCE-AHP, MAVT, SMART,
                                      ARAS, WASPAS)
"""

# ============================================================
# DATA DASAR (identik dengan Bab 1-4, JANGAN diubah)
# ============================================================
vendor = [
    "Nirmala Catering", "Cita Rasa Boga", "Sajian Prima", "Meja Utama",
    "Berkah Boga", "Dapur Bunda", "Rasa Nusantara", "Prima Boga",
    "Sinar Catering", "Boga Sejahtera",
]

kriteria = [
    "Kualitas Rasa", "Ketepatan Waktu", "Variasi Menu", "Rating Kepuasan",
    "Harga/Porsi", "Lead Time", "Jml Komplain",
]

tipe = ["benefit", "benefit", "benefit", "benefit", "cost", "cost", "cost"]

data = {
    "Nirmala Catering": [85, 78, 70, 80, 35000, 3, 2],
    "Cita Rasa Boga":   [90, 70, 65, 75, 42000, 2, 1],
    "Sajian Prima":     [75, 85, 80, 85, 38000, 4, 3],
    "Meja Utama":       [80, 80, 75, 82, 36000, 3, 1],
    "Berkah Boga":      [70, 90, 85, 78, 30000, 5, 4],
    "Dapur Bunda":      [78, 75, 60, 70, 28000, 3, 2],
    "Rasa Nusantara":   [88, 65, 90, 88, 45000, 2, 1],
    "Prima Boga":       [72, 82, 55, 65, 27000, 4, 3],
    "Sinar Catering":   [83, 88, 70, 80, 33000, 3, 1],
    "Boga Sejahtera":   [68, 60, 50, 60, 25000, 5, 5],
}

bobot_bab2 = [0.20, 0.15, 0.10, 0.15, 0.20, 0.10, 0.10]
n = len(kriteria)


def normalisasi(matriks_data):
    maks = [max(matriks_data[v][j] for v in vendor) for j in range(n)]
    mins = [min(matriks_data[v][j] for v in vendor) for j in range(n)]
    hasil = {}
    for v in vendor:
        baris = []
        for j in range(n):
            if tipe[j] == "benefit":
                baris.append(round(matriks_data[v][j] / maks[j], 4))
            else:
                baris.append(round(mins[j] / matriks_data[v][j], 4))
        hasil[v] = baris
    return hasil


def hitung_skor(matriks_normal, bobot):
    return {v: round(sum(matriks_normal[v][j] * bobot[j] for j in range(n)), 4) for v in vendor}


def cetak_peringkat(judul, skor):
    print("\n" + "-" * 70)
    print(judul)
    print("-" * 70)
    for i, v in enumerate(sorted(vendor, key=lambda x: skor[x], reverse=True), 1):
        print(f"{i:>2}. {v:<18} skor = {skor[v]}")


matriks_normal = normalisasi(data)
skor_saw = hitung_skor(matriks_normal, bobot_bab2)

# ============================================================
# BAGIAN A: ARAS - VARIAN PRAKTIS (S0 = 1)
# ============================================================
print("=" * 70)
print("BAGIAN A: ARAS (VARIAN PRAKTIS - memakai matriks ternormalisasi SAW)")
print("=" * 70)
print("Alternatif optimal: seluruh kriteria bernilai 1 (terbaik) setelah normalisasi")

S0 = sum(1 * bobot_bab2[j] for j in range(n))
S_i = hitung_skor(matriks_normal, bobot_bab2)  # sama persis dengan skor_saw
K_i = {v: round(S_i[v] / S0, 4) for v in vendor}

print(f"\nS0 (skor alternatif optimal) = {S0}")
print(f"\n{'Vendor':<20}{'S_i':>10}{'S0':>8}{'K_i (ARAS)':>12}")
for v in vendor:
    print(f"{v:<20}{S_i[v]:>10}{S0:>8}{K_i[v]:>12}")

rank_saw = sorted(vendor, key=lambda v: skor_saw[v], reverse=True)
rank_aras = sorted(vendor, key=lambda v: K_i[v], reverse=True)
print(f"\n>>> Urutan ARAS (varian praktis) identik dengan SAW: {rank_saw == rank_aras} <<<")

# ============================================================
# BAGIAN A2: ARAS - VARIAN ASLI ZAVADSKAS (normalisasi jumlah kolom)
# ============================================================
print("\n" + "=" * 70)
print("BAGIAN A2: ARAS (VARIAN ASLI - normalisasi jumlah kolom, untuk kejujuran metodologis)")
print("=" * 70)

optimal = []
for j in range(n):
    kolom = [data[v][j] for v in vendor]
    optimal.append(max(kolom) if tipe[j] == "benefit" else min(kolom))

semua_baris = {"OPTIMAL": optimal}
semua_baris.update(data)

# transformasi cost -> reciprocal (1/x), benefit tetap
transformasi = {}
for nama, baris in semua_baris.items():
    transformasi[nama] = [baris[j] if tipe[j] == "benefit" else 1 / baris[j] for j in range(n)]

jumlah_kolom = [sum(transformasi[nm][j] for nm in transformasi) for j in range(n)]
normal_asli = {nm: [transformasi[nm][j] / jumlah_kolom[j] for j in range(n)] for nm in transformasi}

S_asli = {nm: sum(normal_asli[nm][j] * bobot_bab2[j] for j in range(n)) for nm in transformasi}
S0_asli = S_asli["OPTIMAL"]
K_asli = {v: round(S_asli[v] / S0_asli, 4) for v in vendor}

print(f"S0 (varian asli) = {round(S0_asli,6)}")
cetak_peringkat("Peringkat ARAS (varian asli Zavadskas)", K_asli)

rank_asli = sorted(vendor, key=lambda v: K_asli[v], reverse=True)
print(f"\n>>> Urutan ARAS (varian asli) identik dengan SAW: {rank_saw == rank_asli} <<<")
if rank_saw != rank_asli:
    print(">>> Catatan: varian asli menghasilkan pergeseran kecil di posisi Meja Utama")
    print(">>> dan Cita Rasa Boga, karena skema normalisasinya berbeda (jumlah kolom, bukan maks/min).")

# ============================================================
# BAGIAN B: WASPAS (lambda = 1)
# ============================================================
print("\n" + "=" * 70)
print("BAGIAN B: WASPAS PADA LAMBDA = 1 (IDENTITAS ALJABAR DENGAN SAW)")
print("=" * 70)

lam = 1.0
# P_i^WPM tidak perlu dihitung karena koefisiennya (1-lambda) = 0
skor_waspas = {v: round(lam * skor_saw[v] + (1 - lam) * 0, 4) for v in vendor}

print(f"Q_i = {lam} x V_i^SAW + {round(1-lam,2)} x P_i^WPM  ->  suku WPM lenyap (dikali 0)")
print(f"\n{'Vendor':<20}{'Skor SAW':>12}{'Skor WASPAS':>14}{'Selisih':>10}")
for v in vendor:
    selisih = round(skor_saw[v] - skor_waspas[v], 6)
    print(f"{v:<20}{skor_saw[v]:>12}{skor_waspas[v]:>14}{selisih:>10}")
print("\n>>> Selisih SAW vs WASPAS(lambda=1) = 0 untuk seluruh vendor. <<<")

# ============================================================
# BAGIAN C: PAPAN SKOR KUMULATIF (6 METODE)
# ============================================================
print("\n" + "=" * 70)
print("BAGIAN C: PAPAN SKOR KUMULATIF - 6 METODE")
print("=" * 70)

bobot_ahp = [0.2061, 0.1445, 0.0996, 0.1445, 0.2061, 0.0996, 0.0996]
skor_mce = hitung_skor(matriks_normal, bobot_ahp)

swing_poin = [100, 70, 48, 70, 100, 48, 48]
bobot_smart = [round(p / sum(swing_poin), 4) for p in swing_poin]
skor_smart = hitung_skor(matriks_normal, bobot_smart)

skor_mavt = hitung_skor(matriks_normal, bobot_bab2)  # identik SAW

print(f"{'Rank':<6}{'Vendor':<20}{'SAW':>9}{'MCE-AHP':>9}{'MAVT':>9}{'SMART':>9}{'ARAS':>9}{'WASPAS':>9}")
for i, v in enumerate(rank_saw, 1):
    print(f"{i:<6}{v:<20}{skor_saw[v]:>9}{skor_mce[v]:>9}{skor_mavt[v]:>9}"
          f"{skor_smart[v]:>9}{K_i[v]:>9}{skor_waspas[v]:>9}")

semua_sama = all(
    sorted(vendor, key=lambda v: sk[v], reverse=True) == rank_saw
    for sk in [skor_mce, skor_mavt, skor_smart, K_i, skor_waspas]
)
print(f"\n>>> Urutan identik di keenam metode (varian praktis ARAS): {semua_sama} <<<")

BAGIAN A: ARAS (VARIAN PRAKTIS - memakai matriks ternormalisasi SAW)
Alternatif optimal: seluruh kriteria bernilai 1 (terbaik) setelah normalisasi

S0 (skor alternatif optimal) = 1.0

Vendor                     S_i      S0  K_i (ARAS)
Nirmala Catering        0.7926     1.0      0.7926
Cita Rasa Boga          0.8358     1.0      0.8358
Sajian Prima             0.757     1.0       0.757
Meja Utama              0.8398     1.0      0.8398
Berkah Boga             0.7646     1.0      0.7646
Dapur Bunda             0.7796     1.0      0.7796
Rasa Nusantara           0.865     1.0       0.865
Prima Boga              0.7371     1.0      0.7371
Sinar Catering          0.8634     1.0      0.8634
Boga Sejahtera           0.669     1.0       0.669

>>> Urutan ARAS (varian praktis) identik dengan SAW: True <<<

BAGIAN A2: ARAS (VARIAN ASLI - normalisasi jumlah kolom, untuk kejujuran metodologis)
S0 (varian asli) = 0.113652

----------------------------------------------------------------------
Perin

In [ ]:
"""
BAB 6 - Kapan Hasilnya Mulai Berbeda: WPM, TOPSIS Sekilas
Studi Kasus Baku Buku: Pemilihan Vendor Katering (10 vendor x 7 kriteria)

Alur:
  Bagian A - WPM      : skor perkalian berbobot (P_i = prod r_ij^wj),
                        dibandingkan dengan SAW
  Bagian B - TOPSIS   : normalisasi vektor, solusi ideal positif/negatif,
                        jarak Euclidean, koefisien kedekatan C_i
  Bagian C - Rangkuman: papan skor kumulatif 8 metode + klasifikasi
"""

import math

# ============================================================
# DATA DASAR (identik dengan Bab 1-5, JANGAN diubah)
# ============================================================
vendor = [
    "Nirmala Catering", "Cita Rasa Boga", "Sajian Prima", "Meja Utama",
    "Berkah Boga", "Dapur Bunda", "Rasa Nusantara", "Prima Boga",
    "Sinar Catering", "Boga Sejahtera",
]

kriteria = [
    "Kualitas Rasa", "Ketepatan Waktu", "Variasi Menu", "Rating Kepuasan",
    "Harga/Porsi", "Lead Time", "Jml Komplain",
]

tipe = ["benefit", "benefit", "benefit", "benefit", "cost", "cost", "cost"]

data = {
    "Nirmala Catering": [85, 78, 70, 80, 35000, 3, 2],
    "Cita Rasa Boga":   [90, 70, 65, 75, 42000, 2, 1],
    "Sajian Prima":     [75, 85, 80, 85, 38000, 4, 3],
    "Meja Utama":       [80, 80, 75, 82, 36000, 3, 1],
    "Berkah Boga":      [70, 90, 85, 78, 30000, 5, 4],
    "Dapur Bunda":      [78, 75, 60, 70, 28000, 3, 2],
    "Rasa Nusantara":   [88, 65, 90, 88, 45000, 2, 1],
    "Prima Boga":       [72, 82, 55, 65, 27000, 4, 3],
    "Sinar Catering":   [83, 88, 70, 80, 33000, 3, 1],
    "Boga Sejahtera":   [68, 60, 50, 60, 25000, 5, 5],
}

bobot = [0.20, 0.15, 0.10, 0.15, 0.20, 0.10, 0.10]
n = len(kriteria)


def normalisasi(matriks_data):
    maks = [max(matriks_data[v][j] for v in vendor) for j in range(n)]
    mins = [min(matriks_data[v][j] for v in vendor) for j in range(n)]
    hasil = {}
    for v in vendor:
        baris = []
        for j in range(n):
            if tipe[j] == "benefit":
                baris.append(matriks_data[v][j] / maks[j])
            else:
                baris.append(mins[j] / matriks_data[v][j])
        hasil[v] = baris
    return hasil


matriks_normal = normalisasi(data)
skor_saw = {v: round(sum(matriks_normal[v][j] * bobot[j] for j in range(n)), 4) for v in vendor}
rank_saw = sorted(vendor, key=lambda v: skor_saw[v], reverse=True)


def cetak_rank(judul, skor):
    print("\n" + "-" * 70)
    print(judul)
    print("-" * 70)
    for i, v in enumerate(sorted(vendor, key=lambda x: skor[x], reverse=True), 1):
        print(f"{i:>2}. {v:<18} skor = {round(skor[v],4)}")


# ============================================================
# BAGIAN A: WPM (Weighted Product Model)
# ============================================================
print("=" * 70)
print("BAGIAN A: WPM - SKOR PERKALIAN BERBOBOT")
print("=" * 70)
print("Formula: P_i = produk(r_ij ^ w_j) untuk j=1..7")

skor_wpm = {}
for v in vendor:
    ln_sum = sum(bobot[j] * math.log(matriks_normal[v][j]) for j in range(n))
    skor_wpm[v] = round(math.exp(ln_sum), 4)

print(f"\n{'Vendor':<20}{'Skor SAW':>12}{'Skor WPM':>12}")
for v in vendor:
    print(f"{v:<20}{skor_saw[v]:>12}{skor_wpm[v]:>12}")

cetak_rank("Peringkat WPM (setelah diurutkan ulang)", skor_wpm)

rank_wpm = sorted(vendor, key=lambda v: skor_wpm[v], reverse=True)
print(f"\nPerbandingan posisi SAW vs WPM:")
berubah_wpm = 0
for i in range(len(vendor)):
    v_saw, v_wpm = rank_saw[i], rank_wpm[i]
    status = "Tetap" if v_saw == v_wpm else "Berubah"
    if v_saw != v_wpm:
        berubah_wpm += 1
    print(f"  posisi {i+1}: SAW={v_saw:<20} WPM={v_wpm:<20} [{status}]")
print(f"\n>>> Jumlah posisi berbeda antara SAW dan WPM: {berubah_wpm} dari 10 <<<")

# ============================================================
# BAGIAN B: TOPSIS
# ============================================================
print("\n" + "=" * 70)
print("BAGIAN B: TOPSIS - JARAK KE SOLUSI IDEAL")
print("=" * 70)
print("Normalisasi vektor: r_ij = x_ij / sqrt(sum_i x_ij^2)")

denom = [math.sqrt(sum(data[v][j] ** 2 for v in vendor)) for j in range(n)]
r_vektor = {v: [data[v][j] / denom[j] for j in range(n)] for v in vendor}

# terbobot
v_mat = {v: [r_vektor[v][j] * bobot[j] for j in range(n)] for v in vendor}

# solusi ideal positif (A+) dan negatif (A-)
ideal_pos, ideal_neg = [], []
for j in range(n):
    kolom = [v_mat[v][j] for v in vendor]
    if tipe[j] == "benefit":
        ideal_pos.append(max(kolom))
        ideal_neg.append(min(kolom))
    else:
        ideal_pos.append(min(kolom))
        ideal_neg.append(max(kolom))

D_pos = {v: math.sqrt(sum((v_mat[v][j] - ideal_pos[j]) ** 2 for j in range(n))) for v in vendor}
D_neg = {v: math.sqrt(sum((v_mat[v][j] - ideal_neg[j]) ** 2 for j in range(n))) for v in vendor}
skor_topsis = {v: round(D_neg[v] / (D_pos[v] + D_neg[v]), 4) for v in vendor}

print(f"\n{'Vendor':<20}{'D+':>10}{'D-':>10}{'C_i':>10}")
for v in vendor:
    print(f"{v:<20}{round(D_pos[v],4):>10}{round(D_neg[v],4):>10}{skor_topsis[v]:>10}")

cetak_rank("Peringkat TOPSIS", skor_topsis)

rank_topsis = sorted(vendor, key=lambda v: skor_topsis[v], reverse=True)
print(f"\nPerbandingan posisi SAW vs TOPSIS:")
berubah_topsis = 0
for i in range(len(vendor)):
    v_saw, v_top = rank_saw[i], rank_topsis[i]
    status = "Tetap" if v_saw == v_top else "Berubah"
    if v_saw != v_top:
        berubah_topsis += 1
    print(f"  posisi {i+1}: SAW={v_saw:<20} TOPSIS={v_top:<20} [{status}]")
print(f"\n>>> Jumlah posisi berbeda antara SAW dan TOPSIS: {berubah_topsis} dari 10 <<<")

# ============================================================
# BAGIAN C: RANGKUMAN 8 METODE
# ============================================================
print("\n" + "=" * 70)
print("BAGIAN C: PAPAN SKOR KUMULATIF - 8 METODE")
print("=" * 70)

bobot_ahp = [0.2061, 0.1445, 0.0996, 0.1445, 0.2061, 0.0996, 0.0996]
skor_mce = {v: round(sum(matriks_normal[v][j] * bobot_ahp[j] for j in range(n)), 4) for v in vendor}

swing_poin = [100, 70, 48, 70, 100, 48, 48]
bobot_smart = [round(p / sum(swing_poin), 4) for p in swing_poin]
skor_smart = {v: round(sum(matriks_normal[v][j] * bobot_smart[j] for j in range(n)), 4) for v in vendor}

skor_mavt = skor_saw  # identik
skor_aras = skor_saw  # identik (varian praktis, S0=1)
skor_waspas = skor_saw  # identik (lambda=1)

print(f"{'Rank':<6}{'Vendor':<18}{'SAW':>8}{'MCE':>8}{'MAVT':>8}{'SMART':>8}"
      f"{'ARAS':>8}{'WASPAS':>8}{'WPM':>8}{'TOPSIS':>8}")
for i, v in enumerate(rank_saw, 1):
    print(f"{i:<6}{v:<18}{skor_saw[v]:>8}{skor_mce[v]:>8}{skor_mavt[v]:>8}{skor_smart[v]:>8}"
          f"{skor_aras[v]:>8}{skor_waspas[v]:>8}{skor_wpm[v]:>8}{skor_topsis[v]:>8}")

print("\nKlasifikasi terhadap SAW:")
print(f"  Selalu identik      : MAVT, ARAS, WASPAS (0 dari 10 posisi berubah)")
print(f"  Hampir selalu sama  : MCE-AHP, SMART (0 dari 10 posisi berubah pada kasus ini)")
print(f"  Bisa menyimpang     : WPM ({berubah_wpm} dari 10 posisi berubah)")
print(f"  Bisa menyimpang jauh: TOPSIS ({berubah_topsis} dari 10 posisi berubah)")

BAGIAN A: WPM - SKOR PERKALIAN BERBOBOT
Formula: P_i = produk(r_ij ^ w_j) untuk j=1..7

Vendor                  Skor SAW    Skor WPM
Nirmala Catering          0.7926      0.7792
Cita Rasa Boga            0.8358      0.8204
Sajian Prima               0.757      0.7226
Meja Utama                0.8398      0.8323
Berkah Boga               0.7646      0.7112
Dapur Bunda               0.7796      0.7685
Rasa Nusantara             0.865      0.8429
Prima Boga                0.7371      0.7062
Sinar Catering            0.8634      0.8564
Boga Sejahtera            0.6689      0.6153

----------------------------------------------------------------------
Peringkat WPM (setelah diurutkan ulang)
----------------------------------------------------------------------
 1. Sinar Catering     skor = 0.8564
 2. Rasa Nusantara     skor = 0.8429
 3. Meja Utama         skor = 0.8323
 4. Cita Rasa Boga     skor = 0.8204
 5. Nirmala Catering   skor = 0.7792
 6. Dapur Bunda        skor = 0.7685
 7. Sajian P